# Project 01 – Data Cleaning on an E-commerce Dataset

Note: the raw file (`First_Dataset.xlsx`) is actually a **customer-level summary** table (one row per customer, with aggregated fields like `purchase_count`, `avg_order_value`, `total_spending`) rather than an order-level transaction table (order_id, product, quantity, order_date...) as the assignment PDF describes. The cleaning workflow and the six required questions were adapted to the columns that actually exist in this file.

## 1. Load raw data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_excel('/content/First Dataset.xlsx')
print(df.shape)
df.head()


(61, 17)


,customer_id,first_name,gender,age,city,province,signup_date,membership_tier,purchase_count,avg_order_value,total_spending,last_purchase_days,payment_method,device,discount_used,returned_items,satisfaction_score
0,1001,Reza,F,19.0,Karaj,Alborz,2025-02-19,VIP,17,121.53,2066.01,16,Card,Android,Yes,3,5
1,1002,Sina,M,53.0,Tehran,Tehran,2022-08-19,Gold,12,326.47,3917.64,3,Card,Web,No,5,3
2,1003,Parsa,F,31.0,Shiraz,Fars,2023-06-20,Gold,21,59.46,1248.66,22,Online Wallet,iPhone,Yes,6,1
3,1004,Sina,F,58.0,Mashhad,Khorasan,2021-11-08,Gold,23,266.15,6121.45,40,Card,Android,No,4,4
4,1005,Kimia,M,28.0,Isfahan,Isfahan,2021-10-21,Silver,23,169.54,3899.42,273,Online Wallet,Android,Yes,7,4


## 2. Explore data quality issues
Checking nulls, duplicates, value ranges, and cross-column consistency.

In [2]:
print(df.isnull().sum())
print('\nFull duplicate rows:', df.duplicated().sum())
print('\nAge range:', df['age'].min(), '-', df['age'].max())


customer_id           0
first_name            0
gender                0
age                   1
city                  0
province              0
signup_date           0
membership_tier       0
purchase_count        0
avg_order_value       0
total_spending        1
last_purchase_days    0
payment_method        0
device                0
discount_used         0
returned_items        0
satisfaction_score    0
dtype: int64

Full duplicate rows: 1

Age range: 19.0 - 145.0


## 3. Clean the data

**Issues found and how they were fixed:**

1. **Duplicate row** - customer 1014 appeared twice with identical values → dropped the repeat.
2. **Impossible age (145)** - no human is 145 years old → treated as invalid, set to missing, then imputed with the median age.
3. **Missing `age`** (1 row) - imputed with the median age.
4. **Missing `total_spending`** (1 row) - recomputed as `avg_order_value * purchase_count`, since this identity holds exactly for every other row in the dataset.
5. **Inconsistent `total_spending`** - customer 1030 showed 25000 vs an expected ~4079 (`avg_order_value * purchase_count`) → replaced with the consistent, recomputed value.
6. **`returned_items` > `purchase_count`** (6 rows) - logically impossible (can't return more items than purchased) → capped at `purchase_count`.
7. **`gender` inconsistent with `first_name`** (38 of 60 rows!) - these are common Persian names with a strongly conventional gender (Ali/Amir/Arash/Reza/Sina/Parsa → male, Kimia/Maryam/Mina/Neda/Sara/Zahra → female). Most rows had the *opposite* gender recorded. Standardized gender using first_name.
8. **Data types** - `signup_date` → datetime, `age` → int, `discount_used` → boolean (Yes/No → True/False).

In [3]:
log = []

# duplicates
before = len(df)
df = df.drop_duplicates()
log.append(f"Removed {before-len(df)} fully duplicated row(s).")

# impossible age
mask_bad_age = df['age'] > 100
df.loc[mask_bad_age, 'age'] = np.nan
log.append(f"Flagged {mask_bad_age.sum()} impossible age value(s) as invalid.")

# missing age -> median impute
median_age = df['age'].median()
n_missing_age = df['age'].isna().sum()
df['age'] = df['age'].fillna(median_age)
log.append(f"Filled {n_missing_age} missing/invalid age value(s) with median age ({median_age:.0f}).")

# missing total_spending -> recompute
recompute_mask = df['total_spending'].isna()
df.loc[recompute_mask, 'total_spending'] = df.loc[recompute_mask, 'avg_order_value'] * df.loc[recompute_mask, 'purchase_count']
log.append(f"Recomputed {recompute_mask.sum()} missing total_spending value(s).")

# inconsistent total_spending -> fix
expected = df['avg_order_value'] * df['purchase_count']
inconsistent = ((df['total_spending'] - expected).abs() / expected.replace(0, np.nan) > 0.05).fillna(False)
df.loc[inconsistent, 'total_spending'] = expected[inconsistent]
log.append(f"Corrected {inconsistent.sum()} total_spending value(s) inconsistent with avg_order_value*purchase_count.")

# returned_items > purchase_count -> cap
bad_returns = df['returned_items'] > df['purchase_count']
df.loc[bad_returns, 'returned_items'] = df.loc[bad_returns, 'purchase_count']
log.append(f"Capped {bad_returns.sum()} returned_items value(s) exceeding purchase_count.")

# gender inconsistent with first_name -> correct using name convention
name_gender_map = {
    'Ali':'M','Amir':'M','Arash':'M','Reza':'M','Sina':'M','Parsa':'M',
    'Kimia':'F','Maryam':'F','Mina':'F','Neda':'F','Sara':'F','Zahra':'F'
}
expected_gender = df['first_name'].map(name_gender_map)
mismatch = df['gender'] != expected_gender
df.loc[mismatch, 'gender'] = expected_gender[mismatch]
log.append(f"Corrected {mismatch.sum()} gender value(s) inconsistent with the conventional gender of first_name.")

# types
df['signup_date'] = pd.to_datetime(df['signup_date'])
df['age'] = df['age'].astype(int)
df['discount_used'] = df['discount_used'].map({'Yes': True, 'No': False})
log.append("Standardized data types (signup_date->datetime, age->int, discount_used->bool).")

df = df.reset_index(drop=True)
for l in log:
    print('-', l)


- Removed 1 fully duplicated row(s).
- Flagged 1 impossible age value(s) as invalid.
- Filled 2 missing/invalid age value(s) with median age (45).
- Recomputed 1 missing total_spending value(s).
- Corrected 1 total_spending value(s) inconsistent with avg_order_value*purchase_count.
- Capped 6 returned_items value(s) exceeding purchase_count.
- Corrected 38 gender value(s) inconsistent with the conventional gender of first_name.
- Standardized data types (signup_date->datetime, age->int, discount_used->bool).


In [4]:
# sanity check: gender should now be fully consistent with first_name
df.groupby(['first_name','gender']).size().unstack(fill_value=0)


gender,F,M
first_name,,
Ali,0,6
Amir,0,6
Arash,0,4
Kimia,7,0
Maryam,4,0
Mina,3,0
Neda,8,0
Parsa,0,4
Reza,0,7


In [5]:
df.to_excel('cleaned_dataset.xlsx', index=False)
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   customer_id         60 non-null     int64         
 1   first_name          60 non-null     object        
 2   gender              60 non-null     object        
 3   age                 60 non-null     int64         
 4   city                60 non-null     object        
 5   province            60 non-null     object        
 6   signup_date         60 non-null     datetime64[ns]
 7   membership_tier     60 non-null     object        
 8   purchase_count      60 non-null     int64         
 9   avg_order_value     60 non-null     float64       
 10  total_spending      60 non-null     float64       
 11  last_purchase_days  60 non-null     int64         
 12  payment_method      60 non-null     object        
 13  device              60 non-null     object        
 

## 4. Answering the project's questions

**Q: Which cities generated the highest revenue?**

In [6]:
df.groupby('city')['total_spending'].sum().sort_values(ascending=False)

,total_spending
city,
Mashhad,43197.58
Tabriz,36192.49
Karaj,26189.81
Ahvaz,23384.78
Isfahan,21975.32
Shiraz,20715.92
Rasht,15168.54
Tehran,15161.29


**Q: Which customers purchased the most?**

In [7]:
df.nlargest(5, 'purchase_count')[['customer_id','first_name','city','purchase_count','total_spending']]

,customer_id,first_name,city,purchase_count,total_spending
5,1006,Amir,Mashhad,35,3868.55
8,1009,Kimia,Karaj,34,11615.42
39,1040,Amir,Tehran,34,1280.44
43,1044,Mina,Shiraz,33,14354.34
13,1014,Reza,Tabriz,31,3353.89


**Q: What is the average order value (overall, and by membership tier)?**

In [8]:
print('Overall:', round(df['avg_order_value'].mean(), 2))
df.groupby('membership_tier')['avg_order_value'].mean().sort_values(ascending=False)


Overall: 213.16


,avg_order_value
membership_tier,
Silver,281.532500
Gold,230.909474
Bronze,203.451667
VIP,165.848000


**Q: Which payment method is used the most?**

In [9]:
df['payment_method'].value_counts()

,count
payment_method,
Online Wallet,23
Cash,19
Card,18


**Q: What effect does using a discount have on spending?**

In [10]:
df.groupby('discount_used')['total_spending'].agg(['mean','median','count'])

,mean,median,count
discount_used,,,
False,3736.741176,2751.72,34
True,2882.174231,1945.17,26


**Q: What time range does the data cover?**
(No `order_date` exists in this file — using `signup_date` as the closest available proxy.)

In [11]:
print(df['signup_date'].min().date(), '→', df['signup_date'].max().date())

2021-02-23 → 2025-09-09


**Q: Was there anything in the data that could lead to wrong conclusions if left uncleaned?**
Yes — see the cleaning log above: 1 duplicate row, 1 impossible age, 1 total_spending far outside what the other columns implied, 6 rows where returns exceeded purchases, and 2 missing values.

## 7. Full Exploratory Data Analysis (EDA)
In this section, we'll perform a deeper dive into the data distribution and relationships.

In [12]:
# 1. Summary Statistics for Numerical Columns
display(df.describe())

,customer_id,age,signup_date,purchase_count,avg_order_value,total_spending,last_purchase_days,returned_items,satisfaction_score
count,60.000000,60.000000,60,60.000000,60.000000,60.000000,60.000000,60.000000,60.000000
mean,1030.500000,43.700000,2023-02-03 14:00:00,17.383333,213.156500,3366.428833,198.600000,3.800000,2.983333
min,1001.000000,19.000000,2021-02-23 00:00:00,0.000000,27.630000,0.000000,3.000000,0.000000,1.000000
25%,1015.750000,31.750000,2021-12-06 12:00:00,10.750000,108.137500,1167.035000,134.250000,1.000000,2.000000
50%,1030.500000,45.000000,2022-08-15 00:00:00,17.000000,165.435000,2128.690000,203.000000,3.500000,3.000000
75%,1045.250000,58.000000,2024-06-14 12:00:00,24.500000,324.107500,4213.117500,273.750000,6.000000,4.000000
max,1060.000000,65.000000,2025-09-09 00:00:00,35.000000,449.810000,14354.340000,365.000000,8.000000,5.000000
std,17.464249,13.946994,NaN,10.149880,130.509923,3235.132979,97.843043,2.666808,1.431979


In [13]:
import plotly.express as px

# 2. Distribution of Key Metrics
fig_age = px.histogram(df, x='age', nbins=20, title='Age Distribution', marginal='box')
fig_age.show()

fig_spend = px.histogram(df, x='total_spending', title='Total Spending Distribution', marginal='violin')
fig_spend.show()

In [14]:
# 3. Categorical Analysis: Membership & Payment Methods
fig_tier = px.pie(df, names='membership_tier', title='Customer Breakdown by Membership Tier', hole=0.4)
fig_tier.show()

fig_pay = px.sunburst(df, path=['membership_tier', 'payment_method'], values='total_spending',
                    title='Spending by Tier and Payment Method')
fig_pay.show()

In [15]:
# 4. Correlation Heatmap
# Selecting numerical columns only
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig_corr = px.imshow(corr, text_auto=True, aspect='auto',
                    title='Correlation Heatmap of Numerical Features',
                    color_continuous_scale='RdBu_r')
fig_corr.show()

In [16]:
# 5. Temporal Trends: Signups over time
signups_over_time = df.groupby(df['signup_date'].dt.to_period('M')).size().reset_index(name='count')
signups_over_time['signup_date'] = signups_over_time['signup_date'].astype(str)

fig_time = px.line(signups_over_time, x='signup_date', y='count', markers=True,
                  title='Customer Sign-ups Over Time (Monthly)',
                  labels={'signup_date': 'Month', 'count': 'Number of Sign-ups'})
fig_time.show()

## 5. Extra questions

**Which membership tier generates the most total revenue - and is 'VIP' actually the most valuable tier?**

In [17]:
print(df.groupby('membership_tier')['total_spending'].sum().sort_values(ascending=False))
print()
df.groupby('membership_tier').agg(customers=('customer_id','count'),
                                    avg_spending=('total_spending','mean'),
                                    avg_order_value=('avg_order_value','mean'),
                                    avg_satisfaction=('satisfaction_score','mean')).sort_values('avg_spending', ascending=False)


membership_tier
Gold      75153.20
Bronze    56999.81
VIP       36393.31
Silver    33439.41
Name: total_spending, dtype: float64



,customers,avg_spending,avg_order_value,avg_satisfaction
membership_tier,,,,
Silver,8,4179.926250,281.532500,2.500000
Gold,19,3955.431579,230.909474,2.736842
Bronze,18,3166.656111,203.451667,3.111111
VIP,15,2426.220667,165.848000,3.400000


**Return rate by device - which platform has the messiest returns?**

In [18]:
tmp = df[df['purchase_count']>0].copy()
tmp['return_rate'] = tmp['returned_items']/tmp['purchase_count']
tmp.groupby('device')['return_rate'].mean().sort_values(ascending=False)


,return_rate
device,
Web,0.411916
Android,0.378864
iPhone,0.242099


**Does return rate or recency of last purchase correlate with satisfaction?**

In [19]:
tmp2 = df[df['purchase_count']>0].copy()
tmp2['return_rate'] = tmp2['returned_items']/tmp2['purchase_count']
tmp2[['satisfaction_score','return_rate','last_purchase_days']].corr()['satisfaction_score']


,satisfaction_score
satisfaction_score,1.000000
return_rate,-0.146023
last_purchase_days,-0.274567


**Churn risk list: valuable customers who haven't purchased in a long time (candidates for a win-back campaign)**

In [20]:
risk = df[(df['last_purchase_days']>250) & (df['total_spending']>df['total_spending'].median())]
risk[['customer_id','first_name','city','total_spending','last_purchase_days','membership_tier']].sort_values('total_spending', ascending=False)


,customer_id,first_name,city,total_spending,last_purchase_days,membership_tier
9,1010,Arash,Karaj,7241.47,276,Silver
34,1035,Arash,Tabriz,5918.21,365,Bronze
52,1053,Sina,Tehran,5805.54,320,Gold
29,1030,Sina,Mashhad,4079.14,340,VIP
4,1005,Kimia,Isfahan,3899.42,273,Silver
53,1054,Reza,Tabriz,3691.44,304,Gold
10,1011,Neda,Karaj,2264.04,272,Bronze
6,1007,Reza,Rasht,2237.35,298,VIP
48,1049,Ali,Mashhad,2140.38,357,Gold


**Does gender correlate with average spending?**
(Using the corrected `gender` column — the original had 38/60 rows with the wrong gender for the customer's name.)

In [21]:
df.groupby('gender')['total_spending'].mean()

,total_spending
gender,
F,3634.781200
M,3174.748571


## 6. Advanced Insights & Interactive Visualizations
Using Plotly for interactive charts and Scikit-Learn for customer segmentation.

In [22]:
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# 1. Interactive Revenue by City
city_rev = df.groupby('city')['total_spending'].sum().reset_index().sort_values('total_spending', ascending=False)
fig1 = px.bar(city_rev, x='city', y='total_spending', color='total_spending',
             title='Total Revenue by City (Interactive)',
             labels={'total_spending':'Total Spending', 'city':'City'})
fig1.show()

### Customer Segmentation (K-Means Clustering)
We will segment customers based on `total_spending`, `purchase_count`, and `age` to identify different archetypes.

In [23]:
# Prepare data for clustering
features = ['age', 'purchase_count', 'total_spending']
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Apply K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled).astype(str)

# Visualize clusters in 3D
fig2 = px.scatter_3d(df, x='age', y='purchase_count', z='total_spending',
                    color='cluster', hover_data=['first_name', 'city', 'membership_tier'],
                    title='Customer Segments (3D K-Means Clustering)')
fig2.show()